# Stage 11 — Lock 2 : ensemble à 3 modèles (pari de diversification)

**Statut : candidat, pas encore verrouillé.** On a 1 mois et 300 soumissions → on teste librement.

**L'idée :** notre Lock 1 (baseline) = 2 modèles (boosting + logistique). Ici on en ajoute un 3e (un boosting plus profond et plus régularisé). Un ensemble plus *diversifié* réduit la variance — et la variance est notre ennemi sur des données décalées.

**Honnêteté :** en CV honnête (monde à 54%), ce modèle fait **+0.004** vs la baseline — un gain réel mais **sous le seuil de bruit** (±0.009). On ne peut donc pas *prouver* qu'il est meilleur. On le soumet pour voir, mais on le garde comme pari diversifié défendable par le raisonnement (diversité → robustesse), pas pour son score public.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
TRAIN_PATH="Train.csv"; TEST_PATH="Test.csv"; SAMPLE_PATH="SampleSubmission.csv"
NODATA=-9999
MONTHS=[f"{i:02d}" for i in range(1,13)]
BANDS=["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
WINTER_BIAS=np.array([.88,.78,.62,.50,.41,.43,.36,.41,.50,.80,.75,.87])

def load_raw(p): return pd.read_csv(p).replace(NODATA, np.nan)
def safe_ratio(a,b): return (a-b)/(a+b)

def build_features(df):
    feats={}; idx={}
    for m in MONTHS:
        g=df[f"green_{m}"]; n=df[f"nir_{m}"]; r=df[f"red_{m}"]; s1=df[f"swir1_{m}"]
        idx.setdefault("NDWI",[]).append(safe_ratio(g,n))
        idx.setdefault("MNDWI",[]).append(safe_ratio(g,s1))
        idx.setdefault("NDVI",[]).append(safe_ratio(n,r))
        idx.setdefault("NDMI",[]).append(safe_ratio(n,s1))
        idx.setdefault("VHmVV",[]).append(df[f"VH_{m}"]-df[f"VV_{m}"])
    def summ(monthly,name):
        a=np.column_stack([np.asarray(x,float) for x in monthly])
        feats[f"{name}_mean"]=np.nanmean(a,1); feats[f"{name}_std"]=np.nanstd(a,1)
        feats[f"{name}_min"]=np.nanmin(a,1);  feats[f"{name}_max"]=np.nanmax(a,1)
        feats[f"{name}_range"]=np.nanmax(a,1)-np.nanmin(a,1); feats[f"{name}_med"]=np.nanmedian(a,1)
        feats[f"{name}_nvalid"]=np.isfinite(a).sum(1)
    for b in BANDS: summ([df[f"{b}_{m}"] for m in MONTHS], b)
    for k,v in idx.items(): summ(v,k)
    return pd.DataFrame(feats,index=df.index).replace([np.inf,-np.inf],np.nan)

def corrupt_realistic(df, seed=0):
    rng=np.random.RandomState(seed); d=df.copy()
    cols={b:[f"{b}_{m}" for m in MONTHS] for b in BANDS}
    arr={b:d[cols[b]].values.astype(float) for b in BANDS}
    for i in range(len(d)):
        miss=rng.rand(12)<WINTER_BIAS*rng.uniform(0.7,1.1)
        for b in BANDS: arr[b][i,miss]=np.nan
    for b in BANDS: d[cols[b]]=arr[b]
    return d

## Entraîner les 3 modèles et prédire

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

train=load_raw(TRAIN_PATH); test=load_raw(TEST_PATH); sample=pd.read_csv(SAMPLE_PATH)
y=train["label"].values
parts=[train]+[corrupt_realistic(train,seed=i+1) for i in range(2)]
aug=pd.concat(parts,ignore_index=True); yaug=np.concatenate([y]*len(parts))
X_aug=build_features(aug); X_test=build_features(test)

# Modele 1: boosting peu profond (notre baseline)
hgb1=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=7,
        l2_regularization=1.0,random_state=42).fit(X_aug,yaug)
# Modele 2: boosting plus profond + plus regularise (la DIVERSITE qu'on ajoute)
hgb2=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,
        l2_regularization=2.0,random_state=7).fit(X_aug,yaug)
# Modele 3: logistique (lineaire, tres different)
lr=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
        LogisticRegression(max_iter=2000)).fit(X_aug,yaug)

# moyenne des 3
proba=(hgb1.predict_proba(X_test)[:,1]+hgb2.predict_proba(X_test)[:,1]+lr.predict_proba(X_test)[:,1])/3
print("ok")

ok


## Soumission + vérifications

In [3]:
submission=pd.DataFrame({"ID":test["ID"],
                         "TargetF1":(proba>=0.5).astype(int),
                         "TargetRAUC":proba})
checks={"colonnes":list(submission.columns)==list(sample.columns),
        "lignes":len(submission)==len(sample),
        "ids":set(submission.ID)==set(sample.ID),
        "f1_binaire":set(submission.TargetF1.unique()).issubset({0,1}),
        "rauc_0_1":submission.TargetRAUC.between(0,1).all(),
        "pas_de_nan":submission.notna().all().all()}
for k,v in checks.items(): print(("OK " if v else "ERREUR"),k)
assert all(checks.values())
taux=submission.TargetF1.mean()
print(f"\nbassins predits: {submission.TargetF1.sum()} ({taux:.1%})  | vrai taux test: 54.4%")
submission.to_csv("submission_lock2.csv", index=False)
print("ecrit: submission_lock2.csv")
submission.head()

OK  colonnes
OK  lignes
OK  ids
OK  f1_binaire
OK  rauc_0_1
OK  pas_de_nan

bassins predits: 576 (55.9%)  | vrai taux test: 54.4%
ecrit: submission_lock2.csv


,ID,TargetF1,TargetRAUC
0,ID_TS_NEW_SBZAYD5I,1,0.997593
1,ID_TS_NEW_7SPRN3PB,1,0.998744
2,ID_TS_NEW_LZOWPHLC,1,0.772676
3,ID_TS_NEW_DN6TUF64,1,0.645272
4,ID_TS_NEW_95N82M49,0,0.000288


## À retenir
- Gain CV = +0.004, **sous le bruit** → on soumet pour voir, sans conclure du score public.
- Pari diversifié défendable : 3 modèles différents = moins de variance = potentiellement plus robuste sur le privé (70%).
- **Règle :** on ne verrouille rien maintenant. On compare le score public à ~0.819, en sachant qu'un écart <0.01 ne veut rien dire.